# 01 - Data Preparation And Token Audit

This notebook documents the dataset state after `scripts/prepare_data.py` has parsed the xLAM source rows, created deterministic splits, and measured Qwen chat-template token lengths.

The notebook intentionally reports aggregate statistics only. It does not print raw gated dataset prompts, tool schemas, or answers.

## Current Decision

- Keep `max_seq_length = 1024` as the first GPU-safe training setting.
- Do not drop only the 3 examples above 2048, because 2048 is not the active limit.
- For SFT at 1024 tokens, exclude over-limit training examples rather than truncating gold assistant answers.
- Do not alter the frozen validation/test split yet. The final test has zero examples above 1024; validation has 5 examples above 1024 and can be evaluated with generation settings, while training exclusion should apply only to train rows.
- Record all exclusions when building the SFT training dataset.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / "data" / "hf_cache"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

from transformers import AutoTokenizer

from tool_calling_lora_dpo.config import load_config
from tool_calling_lora_dpo.data import example_from_json_dict, read_jsonl
from tool_calling_lora_dpo.prompting import token_length

In [2]:
config = load_config(PROJECT_ROOT / "configs" / "experiment.yaml")

{
    "seed": config.seed,
    "model": config.model.id,
    "max_seq_length": config.prompt.max_seq_length,
    "validation_size": config.data.validation_size,
    "test_size": config.data.test_size,
}

{'seed': 42,
 'model': 'Qwen/Qwen2.5-3B-Instruct',
 'max_seq_length': 1024,
 'validation_size': 500,
 'test_size': 300}

In [3]:
manifest_path = PROJECT_ROOT / "data" / "processed" / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["counts"]

{'test': 300, 'total': 60000, 'train': 59200, 'validation': 500}

In [4]:
train_ids = set(manifest["ids"]["train"])
validation_ids = set(manifest["ids"]["validation"])
test_ids = set(manifest["ids"]["test"])

{
    "train_validation": len(train_ids & validation_ids),
    "train_test": len(train_ids & test_ids),
    "validation_test": len(validation_ids & test_ids),
}

{'train_validation': 0, 'train_test': 0, 'validation_test': 0}

In [5]:
processed_dir = PROJECT_ROOT / "data" / "processed"
examples_by_split = {}

for split in ["train", "validation", "test"]:
    rows = read_jsonl(processed_dir / f"{split}.jsonl")
    examples_by_split[split] = [example_from_json_dict(row) for row in rows]
    print(f"{split}: {len(examples_by_split[split])} examples")

train: 59200 examples
validation: 500 examples
test: 300 examples


In [6]:
def percentile(sorted_values: list[int], percent: int) -> int:
    index = round((percent / 100) * (len(sorted_values) - 1))
    return sorted_values[index]


def summarize_lengths(lengths: list[int]) -> dict:
    ordered = sorted(lengths)
    return {
        "p50": percentile(ordered, 50),
        "p90": percentile(ordered, 90),
        "p95": percentile(ordered, 95),
        "p99": percentile(ordered, 99),
        "max": ordered[-1],
    }


tokenizer = AutoTokenizer.from_pretrained(config.model.id)
thresholds = [1024, 1536, 2048]
split_token_audit = {}

for split, examples in examples_by_split.items():
    lengths_by_id = {
        example.source_id: token_length(tokenizer, example, include_assistant=True)
        for example in examples
    }
    lengths = list(lengths_by_id.values())
    split_token_audit[split] = {
        "count": len(lengths),
        "percentiles": summarize_lengths(lengths),
        "over_limit_counts": {
            str(threshold): sum(length > threshold for length in lengths)
            for threshold in thresholds
        },
        "over_1024_ids": [
            source_id for source_id, length in lengths_by_id.items() if length > 1024
        ],
        "over_1536_ids": [
            source_id for source_id, length in lengths_by_id.items() if length > 1536
        ],
        "over_2048_ids": [
            source_id for source_id, length in lengths_by_id.items() if length > 2048
        ],
    }
    print(f"{split} complete")

train complete
validation complete
test complete


In [7]:
summary = {
    split: {
        "count": audit["count"],
        "percentiles": audit["percentiles"],
        "over_limit_counts": audit["over_limit_counts"],
    }
    for split, audit in split_token_audit.items()
}
summary

{'train': {'count': 59200,
  'percentiles': {'p50': 402, 'p90': 678, 'p95': 784, 'p99': 1006, 'max': 2271},
  'over_limit_counts': {'1024': 518, '1536': 33, '2048': 3}},
 'validation': {'count': 500,
  'percentiles': {'p50': 394, 'p90': 644, 'p95': 751, 'p99': 987, 'max': 1478},
  'over_limit_counts': {'1024': 5, '1536': 0, '2048': 0}},
 'test': {'count': 300,
  'percentiles': {'p50': 392, 'p90': 652, 'p95': 804, 'p99': 925, 'max': 1006},
  'over_limit_counts': {'1024': 0, '1536': 0, '2048': 0}}}

In [8]:
output_path = PROJECT_ROOT / "results" / "split_token_audit.json"
output_path.write_text(
    json.dumps(split_token_audit, indent=2, sort_keys=True),
    encoding="utf-8",
)
print(f"Wrote {output_path}")

Wrote C:\\Users\\shiva\\OneDrive\\Desktop\\Project finetune\\results\\split_token_audit.json


## Interpretation

The active limit is 1024 tokens, not 2048. The split-specific audit shows that the 3 examples above 2048 are all in the training split, but the actual 1024-token training issue is larger: 518 training examples exceed the configured limit.

The final test split has zero examples above 1024, so the frozen 300-example test set is compatible with the current setting. The validation split has 5 examples above 1024; because validation is used for development decisions rather than final reported comparison, we can keep it frozen and handle long validation prompts in evaluation logic rather than contaminate the final test.

Recommended next implementation step: build the SFT dataset builder so it excludes only over-1024 training examples, writes an exclusion manifest, and refuses to truncate gold assistant answers silently.